# SVG Scaling Laws — Evaluation v2 (Generation + Evaluation Re-run)

**Runtime**: A100 GPU (Colab Pro)

**Problem**: v1 used `max_new_tokens=512`, but training data median token length is ~1,103.
97.8% of samples were truncated before `</svg>`. This notebook re-runs generation with
`max_new_tokens=4096` (covers p99=4047) and evaluates with 3-tier aggregation.

| # | Section | Est. Time |
|---|---------|----------|
| 1 | Setup (Drive + git pull) | 1 min |
| 2 | Token length statistics | 1 min |
| 3 | Prefix-conditioned sanity check | 1 min |
| 4 | Generation (75 samples) | 10 min |
| 5 | Evaluation | 2 min |
| 6 | Results summary | — |

**Total: ~15 minutes**

All training checkpoints are reused — no retraining needed.

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/svg-scaling-project
!git pull
!pip install -r requirements.txt -q

In [ ]:
import torch, subprocess, os, json

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Checkpoint resolution — v1 used the extended XL checkpoint, so v2 must
# use the same one to isolate the max_new_tokens change.  The directory
# name varies between Colab Drive (mup_xl_extended/, mup/xl/) and local
# rsync copies (mup_scaling_study/xl/), so we try all known locations.
CKPT_CANDIDATES = [
    # 1st priority: extended XL (what v1 actually used)
    'results/runs/mup_xl_extended/best_model.pt',
    # 2nd: µP XL from scaling study (Colab Drive layout)
    'results/runs/mup/xl/best_model.pt',
    # 3rd: µP XL from scaling study (local rsync layout)
    'results/runs/mup_scaling_study/xl/best_model.pt',
]
CKPT = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)
assert CKPT, f'No checkpoint found. Tried:\n  ' + '\n  '.join(CKPT_CANDIDATES)

# Warn if not using the same checkpoint as v1
if 'extended' not in CKPT:
    print(f'[WARN] Extended checkpoint not found, using: {CKPT}')
    print('       old_subset comparison will have a model confound with v1!')
print(f'Checkpoint: {CKPT}')

---
## 2. Token Length Statistics

Measure p50/p90/p95/p99 to validate `max_new_tokens=4096` choice.

In [ ]:
!python scripts/plot_token_histogram.py \
    --data data/tokenized/train.bin \
    --tokenizer tokenizer/bpe_svg.json \
    --output analysis/token_length_histogram.png \
    --stats-output analysis/token_length_stats.json

with open('analysis/token_length_stats.json') as f:
    stats = json.load(f)
print(json.dumps(stats, indent=2))

# Validate our max_new_tokens=4096 covers p99
assert stats['p99'] <= 4096, f"p99={stats['p99']} > 4096, reconsider max_new_tokens"
print(f"\n✓ p99={stats['p99']} <= 4096 — max_new_tokens=4096 is sufficient")

---
## 3. Prefix-Conditioned Sanity Check

Generate 1 sample from `face_partial.svg` prefix to verify the model continues
the prefix rather than re-emitting `<svg`.

In [ ]:
import shutil, glob

test_dir = '/tmp/prefix_test'
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

prefix_text = open('results/prefixes_v2/face_partial.svg').read()
subprocess.run([
    'python', 'src/generate.py',
    '--config', 'configs/xl.yaml',
    '--checkpoint', CKPT,
    '--mup', '--top-k', '0', '--top-p', '0.95',
    '--temperature', '0.8', '--num-samples', '1',
    '--max-tokens', '4096',
    '--prefix', prefix_text,
    '--output-dir', test_dir,
    '--device', 'cuda',
])

# Show result
for f in sorted(glob.glob(f'{test_dir}/*')):
    print(f'\n=== {os.path.basename(f)} ===')
    content = open(f).read()
    print(content[:1000])
    if len(content) > 1000:
        print(f'... ({len(content)} chars total)')

# Check: should NOT start with a new <svg if prefix didn't contain one at position 0
# (our prefix starts with <svg, so model output should continue it, not restart)

---
## 4. Generation (75 samples)

Clean slate, then generate:
- **Unconditional top-p** (0.95): 3 temps × 10 = 30 samples (matches v1 sampling for comparison)
- **Unconditional top-k** (40): 3 temps × 10 = 30 samples (new, for top-p vs top-k comparison)
- **Prefix-conditioned** (top-p 0.95): 3 prefixes × 5 = 15 samples

In [ ]:
import shutil

# Clean slate — prevent stale files from contaminating counts
for d in ['results/samples_v2', 'results/evaluation_v2', 'results/curated_v2']:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'Removed {d}')

def run_generate(extra_args, label):
    """Run generate.py with common args + extra_args."""
    cmd = [
        'python', 'src/generate.py',
        '--config', 'configs/xl.yaml',
        '--checkpoint', CKPT,
        '--mup',
        '--max-tokens', '4096',
        '--device', 'cuda',
    ] + extra_args
    print(f'  {label}...')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  FAILED: {r.stderr[-500:]}')
    return r.returncode

# --- 4a. Unconditional top-p (matches v1: --top-p 0.95) ---
print('=== Unconditional top-p ===')
for temp in ['0.5', '0.8', '1.0']:
    run_generate([
        '--top-k', '0', '--top-p', '0.95',
        '--temperature', temp,
        '--num-samples', '10',
        '--output-dir', f'results/samples_v2/unconditional/topp_t{temp}',
    ], f'top-p temp={temp}')

# --- 4b. Unconditional top-k ---
print('\n=== Unconditional top-k ===')
for temp in ['0.5', '0.8', '1.0']:
    run_generate([
        '--top-k', '40',
        '--temperature', temp,
        '--num-samples', '10',
        '--output-dir', f'results/samples_v2/unconditional/topk_t{temp}',
    ], f'top-k temp={temp}')

# --- 4c. Prefix-conditioned (top-p 0.95, temp=0.8) ---
print('\n=== Prefix-conditioned ===')
for prefix_name in ['face_partial', 'open_path', 'single_shape_group']:
    prefix_text = open(f'results/prefixes_v2/{prefix_name}.svg').read()
    run_generate([
        '--top-k', '0', '--top-p', '0.95',
        '--temperature', '0.8',
        '--num-samples', '5',
        '--prefix', prefix_text,
        '--output-dir', f'results/samples_v2/prefix_conditioned/{prefix_name}',
    ], f'prefix={prefix_name}')

print('\nGeneration complete.')

In [ ]:
# Verify: expect 75 files total (svg + incomplete)
import glob as g

svg_count = len(g.glob('results/samples_v2/**/*.svg', recursive=True))
inc_count = len(g.glob('results/samples_v2/**/*_incomplete.txt', recursive=True))
total = svg_count + inc_count
print(f'Complete SVGs: {svg_count}')
print(f'Incomplete: {inc_count}')
print(f'Total: {total}')
assert total == 75, f'Expected 75 sample files, got {total}'
print('✓ 75 samples generated')

---
## 5. Evaluation

Runs `evaluate.py` on `results/samples_v2/`. Output includes:
- `aggregate_full`: all 75 samples
- `aggregate_old_subset`: unconditional top-p only (30 samples, closest v1 comparator)
- `categories`: per-directory breakdown (top-p vs top-k, per temperature, per prefix)

In [ ]:
subprocess.run([
    'python', 'src/evaluate.py',
    '--config', 'configs/xl.yaml',
    '--checkpoint', CKPT,
    '--mup',
    '--samples-dir', 'results/samples_v2',
    '--test-data', 'data/tokenized/test.bin',
    '--output-dir', 'results/evaluation_v2',
    '--device', 'cuda',
])

---
## 6. Results Summary

In [ ]:
with open('results/evaluation_v2/eval_metrics.json') as f:
    metrics = json.load(f)

def print_agg(label, agg):
    if not agg:
        print(f'  {label}: (empty)')
        return
    n = agg['total_samples']
    print(f'\n  {label} ({n} samples):')
    print(f'    Completion:  {agg["complete_samples"]}/{n} ({agg["completion_rate"]:.1%})')
    print(f'    XML valid:   {agg["xml_valid"]}/{n} ({agg["xml_validity_rate"]:.1%})')
    print(f'    Renders OK:  {agg["render_success"]}/{n} ({agg["render_rate"]:.1%})')
    print(f'    Structural:  {agg["structural_valid"]}/{n} ({agg["structural_validity_rate"]:.1%})')

# Perplexity
if 'perplexity' in metrics:
    ppl = metrics['perplexity']
    print(f'Test perplexity: {ppl["test_perplexity"]:.4f} (loss={ppl["test_loss"]:.4f})')

# Aggregates
samples = metrics.get('samples', {})
print_agg('ALL (aggregate_full)', samples.get('aggregate_full', {}))
print_agg('OLD SUBSET (uncond top-p, v1 comparator)', samples.get('aggregate_old_subset', {}))

# Category breakdown
print('\n  --- Category Breakdown ---')
for cat, agg in sorted(samples.get('categories', {}).items()):
    print_agg(cat, agg)

In [ ]:
# v1 vs v2 comparison (old_subset only — closest apples-to-apples)
v1_path = 'results/evaluation/eval_metrics_v1.json'
if os.path.exists(v1_path):
    with open(v1_path) as f:
        v1 = json.load(f)
    v1s = v1.get('samples', {})
    v2s = samples.get('aggregate_old_subset', {})

    print('=== v1 vs v2 Comparison (unconditional top-p only) ===')
    print(f'  NOTE: v1 had {v1s.get("total_samples", "?")} total samples (uncond+prefix),')
    print(f'        v2 old_subset has {v2s.get("total_samples", "?")} samples (uncond top-p only)')
    print(f'        Prefix samples excluded from v2 subset (content changed)')
    print()
    print(f'  {"Metric":<25s}  {"v1 (all)":>10s}  {"v2 (subset)":>12s}')
    print(f'  {"-"*25}  {"-"*10}  {"-"*12}')
    for key in ['completion_rate', 'xml_validity_rate', 'render_rate', 'structural_validity_rate']:
        v1_val = v1s.get(key, 0)
        v2_val = v2s.get(key, 0)
        print(f'  {key:<25s}  {v1_val:>9.1%}  {v2_val:>11.1%}')
else:
    print('v1 metrics not found — skipping comparison')

In [ ]:
# Show sample grid
from IPython.display import Image, display

grid_path = 'results/evaluation_v2/sample_grid.png'
if os.path.exists(grid_path):
    display(Image(filename=grid_path))
else:
    print('No sample grid generated (no renderable SVGs?)')

---
## Done

Results saved:
- `results/evaluation_v2/eval_metrics.json` — full metrics with 3-tier aggregation
- `results/evaluation_v2/sample_grid.png` — rendered sample grid
- `analysis/token_length_stats.json` — token length statistics
- `analysis/token_length_histogram.png` — histogram plot

Next: `git add` results, push, then write report on local machine.